# Customer Analytics Notebook

## Architecture Overview

This notebook follows a **clean separation of concerns**:

### 🔧 Reusable Transformation Functions
- **Purpose**: Business logic that can be tested, reused, and maintained independently
- **Location**: Cell 3 (Reusable Transformation Functions)
- **Functions**:
  * `load_csv_data()` - Standard CSV loading
  * `convert_date_column()` - Date type conversion
  * `fill_missing_locations()` - Data cleaning for location fields
  * `add_date_features()` - Extract year/month from dates
  * `add_window_rankings()` - Add rank, dense_rank, row_number
  * `filter_by_date()` - Date-based filtering

### 📊 Presentation Logic
- **Purpose**: Display, exploration, and analysis of data
- **Pattern**: Uses transformation functions, then shows/displays results
- **Examples**: `.show()`, `.count()`, `.printSchema()`, aggregations for visualization

### ✅ Benefits
1. **Testability** - Functions can be unit tested independently
2. **Reusability** - Import functions in other notebooks
3. **Maintainability** - Business logic centralized in one place
4. **Readability** - Clear separation between "what to do" and "how to show it"
5. **Parameterization** - Supports widget-based configuration

### Read and Process Data

In [0]:
# Import necessary PySpark libraries
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Initialize Spark session
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
# Reusable transformation functions for customer data processing
# These functions can be imported and reused across notebooks

def convert_date_column(df, column_name, date_format="yyyy-MM-dd"):
    """
    Convert a string column to DateType.
    
    Args:
        df: Input DataFrame
        column_name: Name of the column to convert
        date_format: Date format string (default: yyyy-MM-dd)
    
    Returns:
        DataFrame with converted date column
    """
    return df.withColumn(column_name, to_date(col(column_name), date_format))


def fill_missing_locations(df, default_value="Unknown"):
    """
    Fill missing values in location columns (city, state, country).
    
    Args:
        df: Input DataFrame
        default_value: Value to use for missing data (default: Unknown)
    
    Returns:
        DataFrame with filled location columns
    """
    return df.fillna({
        'city': default_value,
        'state': default_value,
        'country': default_value
    })


def add_date_features(df, date_column):
    """
    Extract year and month from a date column.
    
    Args:
        df: Input DataFrame
        date_column: Name of the date column
    
    Returns:
        DataFrame with additional year and month columns
    """
    return df.withColumn(
        f"{date_column}_year",
        year(col(date_column))
    ).withColumn(
        f"{date_column}_month",
        month(col(date_column))
    )


def add_window_rankings(df, partition_col, order_col, ascending=False):
    """
    Add ranking columns using window functions.
    
    Args:
        df: Input DataFrame
        partition_col: Column to partition by
        order_col: Column to order by
        ascending: Sort order (default: False for descending)
    
    Returns:
        DataFrame with rank, dense_rank, and row_number columns
    """
    order_expr = col(order_col).asc() if ascending else col(order_col).desc()
    window_spec = Window.partitionBy(partition_col).orderBy(order_expr)
    
    return df.withColumn('rank', rank().over(window_spec)) \
             .withColumn('dense_rank', dense_rank().over(window_spec)) \
             .withColumn('row_number', row_number().over(window_spec))


def filter_by_date(df, date_column, start_date):
    """
    Filter DataFrame for records on or after a specific date.
    
    Args:
        df: Input DataFrame
        date_column: Name of the date column to filter on
        start_date: Starting date (string in yyyy-MM-dd format)
    
    Returns:
        Filtered DataFrame
    """
    return df.filter(col(date_column) >= lit(start_date))


def load_csv_data(spark, path, header=True, infer_schema=True):
    """
    Load CSV data with standard options.
    
    Args:
        spark: SparkSession
        path: Path to CSV file
        header: Whether CSV has header row (default: True)
        infer_schema: Whether to infer schema (default: True)
    
    Returns:
        DataFrame loaded from CSV
    """
    return spark.read.csv(path, header=header, inferSchema=infer_schema)


print("✅ Transformation functions loaded successfully")

In [0]:
# Read customers data using reusable function
# Uses parameterized path if widgets are defined, otherwise falls back to default
try:
    customers_path = f'/Volumes/{dbutils.widgets.get("catalog")}/{dbutils.widgets.get("schema")}/{dbutils.widgets.get("volume")}/{dbutils.widgets.get("input_folder")}/customers.csv'
except:
    customers_path = '/Volumes/workspace/default/dataset/raw/customers.csv'

data = load_csv_data(spark, customers_path)
data.show()


In [0]:
# Count total number of customer records (presentation)
data.count()

In [0]:
# Display the schema (presentation)
data.printSchema()

In [0]:
# Apply date conversion transformation
customers_df = convert_date_column(data, "registration_date")

In [0]:
# Apply data cleaning transformation
customers_df = fill_missing_locations(customers_df)


In [0]:
# Apply feature engineering transformation
customers_df = add_date_features(customers_df, "registration_date")
customers_df.show()

In [0]:
# Exploratory Data Analysis: Count unique values in location columns
# This shows the geographic diversity of the customer base
customers_df.select(
    countDistinct("city").alias("unique_cities")
).show()
customers_df.select(
    countDistinct("state").alias("unique_states")
).show()
customers_df.select(
    countDistinct("country").alias("unique_countries")
).show()


In [0]:
# Find top 5 cities with the most customers
customers_df.groupBy('city').count().orderBy(desc('count')).show(5)

In [0]:
# Alternative approach: Top 5 cities by customer count (using col() method)
customers_df.groupBy("city").count().orderBy(col('count').desc()).show(5)

In [0]:
# Find top 5 state-country combinations with most customers
# Useful for understanding geographic distribution patterns
customers_df.groupBy("state", "country").count().orderBy(col('count').desc()).show(5)

In [0]:
# Create a pivot table showing active vs inactive users by state
# Columns will be the unique values from is_active (True/False)
customers_df.groupBy("state").pivot("is_active").count().show()

In [0]:
# Apply window ranking transformation
customers_df = add_window_rankings(customers_df, 'state', 'registration_date')
                

In [0]:
# Display the ranking results to compare different ranking functions
customers_df.select("name", "state", "is_active", "rank", "dense_rank", "row_number").show()

In [0]:
# Apply date filtering transformation
recent_customers = filter_by_date(customers_df, "registration_date", "2025-01-01")
recent_customers.show()

In [0]:
# Count how many customers registered since 2025
recent_customers.count()

In [0]:
# Find the earliest and latest registration dates for each city
# This shows the customer acquisition timeline per location
customers_df.groupBy("city").agg(
    min("registration_date").alias("oldest_customer"), 
    max("registration_date").alias("newest_customer")
).show()

In [0]:
# Save processed DataFrames to Parquet format for efficient storage and future use
# mode="overwrite" replaces existing data if present
output_path = '/Volumes/workspace/default/dataset'
customers_df.write.mode("overwrite").parquet(output_path + "/processed_customers")
recent_customers.write.mode("overwrite").parquet(output_path + "/recent_customers")


### Join `orders_df` with `customers_df`

In [0]:
# Display first 5 rows of the customers DataFrame
customers_df.display(5)

In [0]:
# Read orders data using reusable function
# Uses parameterized path if widgets are defined, otherwise falls back to default
try:
    orders_path = f'/Volumes/{dbutils.widgets.get("catalog")}/{dbutils.widgets.get("schema")}/{dbutils.widgets.get("volume")}/{dbutils.widgets.get("input_folder")}/orders.csv'
except:
    orders_path = '/Volumes/workspace/default/dataset/raw/orders.csv'

orders_df = load_csv_data(spark, orders_path)
orders_df.show(5)

In [0]:
# Apply feature engineering to orders (presentation logic preserved)
orders_df = orders_df.withColumn('order_month', month(col('order_date')))
orders_df.show(5)

In [0]:
# Join customers and orders DataFrames on customer_id (inner join)
# This combines customer information with their order history
customers_orders_df =  customers_df.join(orders_df, 'customer_id', 'inner')
customers_orders_df.show(5)

In [0]:
# Display customers DataFrame again for verification
customers_df.display(5)

In [0]:
# Calculate total number of orders per customer
# Orders are grouped by customer_id and sorted in descending order
customers_orders_count = customers_orders_df.groupBy('customer_id').count().orderBy(col('count').desc())
customers_orders_count.show(10)

In [0]:
# Calculate total spend per customer by summing all order amounts
# Results are sorted to show highest-spending customers first
customer_total_spend = customers_orders_df.groupBy('customer_id').agg(
    sum('total_amount').alias('total_spend')
).orderBy(col('total_spend').desc())
customer_total_spend.show(10)

# Store reference to top customers (for potential further analysis)
top_customers = customer_total_spend

In [0]:
# Calculate average order value per customer
# This shows the typical spending amount per order for each customer
customer_average_spend = customers_orders_df.groupBy('customer_id').agg(
    avg('total_amount').alias('total_spend')
).orderBy(col('total_spend').desc())
customer_average_spend.show(10)

In [0]:
# Count orders by their status (e.g., completed, pending, cancelled)
# This helps understand order fulfillment distribution
order_by_status_count = customers_orders_df.groupBy('status').count().orderBy(col('count').desc())
order_by_status_count.show(10)

# Note: The line below appears incomplete (top_products analysis)
top_products = customers_orders_df.count

In [0]:
# Analyze order volume by month to identify seasonal trends
# Results are ordered chronologically (month 1-12)
order_by_month = customers_orders_df.groupBy('order_month').count().orderBy(col('order_month').asc())
order_by_month.show(10)

In [0]:
# Rank customers by total spend using dense_rank
# Dense rank assigns consecutive ranks without gaps (useful for top-N analysis)
window_spec = Window.orderBy(col('total_spend').desc())
dense_ranked_customers = customer_total_spend.withColumn('dense_rank', dense_rank().over(window_spec))
dense_ranked_customers.show(5)

In [0]:
# Identify customers with high order frequency but low total spending
# These are customers who order often but spend less per order (potential upsell targets)
customer_spend_vs_order = customers_orders_count.join(
    customer_total_spend, 'customer_id', 'inner'
).orderBy(col('count').desc(), col('total_spend'))
customer_spend_vs_order.show(5)